# Abbreviation Variants Generator

This notebook loads the source CSV, detects sentence-local abbreviations in the `question` column, and creates two versions per row:
- `with_abbreviation`
- `without_abbreviation`


In [1]:
# Import regex, path, and pandas libraries.
import re
from pathlib import Path

import pandas as pd


In [2]:
# Set input/output CSV paths and load the question dataset.
PROJECT_ROOT = Path.cwd()  # repository root (the notebooks live here)
# Inputs: example_data/ ships with the repo; point DATA_DIR at your own data folder for full-scale runs.
DATA_DIR = PROJECT_ROOT / "example_data"
ANNOTATION_DIR = DATA_DIR / "Official_annotation"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "abbreviation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

name = 'single_expand_example'
INPUT_CSV = ANNOTATION_DIR / f'{name}.csv'
OUTPUT_CSV = OUTPUT_DIR / f'{name}_with_abbr_variants.csv'

df = pd.read_csv(INPUT_CSV)
df['question'] = df['question'].fillna('').astype(str)
df.head(3)


,Round,Name,Question_id,Rate_quality,question,pmcid,Relevance,Complexity,Confidence,Other_comments
0,NaN,NaN,1,NaN,Can you find structures of Mp1ps ligand bindin...,20053994,NaN,NaN,NaN,NaN
1,NaN,NaN,2,NaN,Can you find structures of Mesorhizobium sp. s...,22745123,NaN,NaN,NaN,NaN
2,NaN,NaN,3,NaN,Can you find structures of TCR-MHC-glycopepti...,10023771,NaN,NaN,NaN,NaN


In [3]:
# Define regex patterns for detecting 'full phrase (ABBR)' and 'ABBR (full phrase)' in text.
# Parentheses content that can include one nested level, e.g. M(pro)
PAREN_CONTENT = r'[^()]+(?:\([^()]*\)[^()]*)*'

# Pattern: "full phrase (ABBR)", where ABBR can be MLNs, SBC, M(pro), YPX(n)L, etc.
ABBR_IN_PARENS = re.compile(
    rf'(?P<full>[A-Za-zα-ωΑ-Ω0-9][A-Za-zα-ωΑ-Ω0-9\-/,\s]{{1,220}}?)\s*'
    rf'\(\s*(?P<abbr>{PAREN_CONTENT})\s*\)'
)
# Pattern: "ABBR (full phrase)" (requires a space before parenthesis)
FULL_IN_PARENS = re.compile(
    rf'(?P<abbr>[A-Za-z0-9][A-Za-z0-9\-]*(?:\([^()]*\)[A-Za-z0-9\-]*)?)\s+'
    rf'\(\s*(?P<full>[A-Za-zα-ωΑ-Ω0-9][A-Za-zα-ωΑ-Ω0-9\-/,\s]{{1,220}}?)\s*\)'
)

FUNCTION_WORDS = {
    'of', 'in', 'with', 'from', 'for', 'to', 'by', 'on', 'at', 'into', 'through',
    'and', 'or', 'the', 'a', 'an', 'its', 'their', 'this', 'that', 'these', 'those',
}


def normalize_spaces(text: str) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    return text


def is_abbreviation_like(text: str) -> bool:
    """Heuristic guard to avoid treating normal parenthetical text as abbreviation."""
    t = normalize_spaces(text)
    if not t or len(t) > 30:
        return False
    if ' ' in t:
        return False
    # Must contain at least one alphabetic character (filters pure numbers like 1, 2+, 420).
    if not re.search(r'[A-Za-z]', t):
        return False
    # Accept uppercase/mixed abbreviations and nested-token forms like M(pro), YPX(n)L.
    return bool(re.search(r'[A-Z]', t) or re.search(r'[A-Za-z]\([^()]+\)', t))


def split_full_phrase(full: str) -> tuple[str, str]:
    """Split a broad captured full phrase into context prefix + core term."""
    words = normalize_spaces(full).split(' ')
    if not words:
        return '', ''

    last_func_idx = -1
    for i, w in enumerate(words):
        if w.lower() in FUNCTION_WORDS:
            last_func_idx = i

    if 0 <= last_func_idx < len(words) - 1:
        prefix = ' '.join(words[: last_func_idx + 1])
        core = ' '.join(words[last_func_idx + 1 :])
        return prefix, core

    return '', ' '.join(words)


def _acronym_candidates(text: str) -> list[str]:
    """Generate suffix phrase candidates from tokenized text."""
    tokens = re.findall(r"[A-Za-z0-9]+(?:[-/][A-Za-z0-9]+)*", text)
    candidates = []
    for i in range(len(tokens)):
        candidates.append(' '.join(tokens[i:]))
    return candidates


def refine_full_name_for_abbr(full_raw: str, abbr: str) -> str:
    """Refine broad full-form capture to a concise term for dictionary extraction."""
    text = normalize_spaces(full_raw)

    # Keep the nearest clause before the abbreviation.
    text = re.split(r'[;:.?!]', text)[-1].strip() or text
    if ',' in text:
        text = text.split(',')[-1].strip() or text

    # Remove cue words often preceding definitions.
    text = re.sub(
        r'^\b(?:termed|called|known\s+as|named|denoted\s+as|abbreviated\s+as|including|such\s+as)\b\s+',
        '',
        text,
        flags=re.IGNORECASE,
    )

    abbr_letters = re.sub(r'[^A-Za-z]', '', abbr).lower()
    if not abbr_letters:
        return text

    letter_options = {abbr_letters}
    if len(abbr_letters) > 1 and abbr_letters.endswith('s'):
        letter_options.add(abbr_letters[:-1])

    best = None
    for cand in _acronym_candidates(text):
        words = cand.split()
        if not words:
            continue
        initials = ''.join(w[0].lower() for w in words if w and w[0].isalnum())
        if initials in letter_options:
            # Prefer shortest phrase matching the acronym initials.
            if best is None or len(words) < len(best.split()):
                best = cand

    if best:
        return normalize_spaces(best)

    # Fallback: remove leading function words.
    words = text.split()
    while words and words[0].lower() in FUNCTION_WORDS:
        words.pop(0)
    return normalize_spaces(' '.join(words) if words else text)


def replace_full_abbr_match(match: re.Match, use_abbreviation: bool) -> str:
    full_raw = normalize_spaces(match.group('full'))
    abbr = normalize_spaces(match.group('abbr'))

    if not is_abbreviation_like(abbr):
        return normalize_spaces(match.group(0))

    prefix, core = split_full_phrase(full_raw)
    core = core or full_raw

    if use_abbreviation:
        replacement = f"{prefix} {abbr}" if prefix else abbr
    else:
        replacement = f"{prefix} {core}" if prefix else core

    return normalize_spaces(replacement)


def replace_abbr_full_match(match: re.Match, use_abbreviation: bool) -> str:
    abbr = normalize_spaces(match.group('abbr'))
    full = normalize_spaces(match.group('full'))

    if not is_abbreviation_like(abbr):
        return normalize_spaces(match.group(0))

    return abbr if use_abbreviation else full


def build_with_abbreviation(sentence: str) -> str:
    # "full (ABBR)" -> "context + ABBR"
    text = ABBR_IN_PARENS.sub(lambda m: replace_full_abbr_match(m, use_abbreviation=True), sentence)
    # "ABBR (full)" -> "ABBR"
    text = FULL_IN_PARENS.sub(lambda m: replace_abbr_full_match(m, use_abbreviation=True), text)
    return normalize_spaces(text)


def build_without_abbreviation(sentence: str) -> str:
    # "full (ABBR)" -> "context + full core"
    text = ABBR_IN_PARENS.sub(lambda m: replace_full_abbr_match(m, use_abbreviation=False), sentence)
    # "ABBR (full)" -> "full"
    text = FULL_IN_PARENS.sub(lambda m: replace_abbr_full_match(m, use_abbreviation=False), text)
    return normalize_spaces(text)


def has_abbreviation_pattern(sentence: str) -> bool:
    for m in ABBR_IN_PARENS.finditer(sentence):
        if is_abbreviation_like(m.group('abbr')):
            return True
    for m in FULL_IN_PARENS.finditer(sentence):
        if is_abbreviation_like(m.group('abbr')):
            return True
    return False


def build_variants(sentence: str):
    sentence = normalize_spaces(sentence)
    detected = has_abbreviation_pattern(sentence)

    if not detected:
        return {
            'with_abbreviation': sentence,
            'without_abbreviation': sentence,
            'has_detected_abbreviation': False,
            'conflict_abbreviations': [],
        }

    return {
        'with_abbreviation': build_with_abbreviation(sentence),
        'without_abbreviation': build_without_abbreviation(sentence),
        'has_detected_abbreviation': True,
        'conflict_abbreviations': [],
    }


In [4]:
# Sanity-check the abbreviation detection patterns on a few hand-crafted examples.
# Quick sanity check on a few examples before processing the full dataset
test_cases = [
    "Can you find structures of aldo-keto reductases (AKRs) showing positive cooperativity with substrates?",
    "Can you find structures of BmmI in complex with malonylated/succinylated macrolactins (MLNs)?",
    "Can you find structures of SARS coronavirus main proteinase (M(pro)) at different pH conditions?",
    "Can you find structures of heliorhodopsins (HeRs) with a cluster of water molecules in the Schiff base cavity (SBC)?",
    "Can you find structures of HAdV-D fiber-knob proteins?",
]
for t in test_cases:
    v = build_variants(t)
    print(f"ORIGINAL:  {t}")
    print(f"ABBR ONLY: {v['with_abbreviation']}")
    print(f"FULL ONLY: {v['without_abbreviation']}")
    print(f"DETECTED:  {v['has_detected_abbreviation']}")
    print()

ORIGINAL:  Can you find structures of aldo-keto reductases (AKRs) showing positive cooperativity with substrates?
ABBR ONLY: Can you find structures of AKRs showing positive cooperativity with substrates?
FULL ONLY: Can you find structures of aldo-keto reductases showing positive cooperativity with substrates?
DETECTED:  True

ORIGINAL:  Can you find structures of BmmI in complex with malonylated/succinylated macrolactins (MLNs)?
ABBR ONLY: Can you find structures of BmmI in complex with MLNs?
FULL ONLY: Can you find structures of BmmI in complex with malonylated/succinylated macrolactins?
DETECTED:  True

ORIGINAL:  Can you find structures of SARS coronavirus main proteinase (M(pro)) at different pH conditions?
ABBR ONLY: Can you find structures of M(pro) at different pH conditions?
FULL ONLY: Can you find structures of SARS coronavirus main proteinase at different pH conditions?
DETECTED:  True

ORIGINAL:  Can you find structures of heliorhodopsins (HeRs) with a cluster of water mole

In [5]:
# Apply build_variants to each question and preview rows where abbreviations were detected.
variants = df['question'].apply(build_variants).apply(pd.Series)
result_df = pd.concat([df, variants], axis=1)

preview = result_df[result_df['has_detected_abbreviation']][
    ['question', 'with_abbreviation', 'without_abbreviation']
]#.head(15)
pd.set_option('display.max_colwidth', 120)
preview

,question,with_abbreviation,without_abbreviation
0,Can you find structures of Mp1ps ligand binding domain 2 (LBD2)?,Can you find structures of LBD2?,Can you find structures of Mp1ps ligand binding domain 2?
1,Can you find structures of Mesorhizobium sp. strain LUK S-selective transaminase (MesAT) in complex with various β-a...,Can you find structures of Mesorhizobium sp. MesAT in complex with various β-amino acids?,Can you find structures of Mesorhizobium sp. strain LUK S-selective transaminase in complex with various β-amino acids?
4,Can you find structures of cyclic nucleotide-gated (CNG) or hyperpolarization-activated cyclic nucleotide-regulated...,Can you find structures of CNG or HCN ion channels?,Can you find structures of cyclic nucleotide-gated or hyperpolarization-activated cyclic nucleotide-regulated ion ch...


In [6]:
# Print summary counts: total rows, rows with detected abbreviations, and rows with confident mappings.
print('Total rows:', len(result_df))
print('Rows with detected abbreviations:', int(result_df['has_detected_abbreviation'].sum()))
print('Rows with mapping conflicts:', int(result_df['conflict_abbreviations'].map(len).gt(0).sum()))

debug_conflicts = result_df[result_df['conflict_abbreviations'].map(len) > 0][['question', 'conflict_abbreviations']]
debug_conflicts.head(20)


Total rows: 5
Rows with detected abbreviations: 3
Rows with mapping conflicts: 0


,question,conflict_abbreviations


In [7]:
# Save the expanded dataset with abbreviation variant columns to the output CSV.
result_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')


Saved: /home/kding1/projects/RAD_PROTEIN/outputs/abbreviation/single_expand_example_with_abbr_variants.csv


## Generate abbreviation-full name dictionary

In [8]:
# Load all PubMed abstracts and build a global abbreviation → full-name frequency dict using regex.
# Build a global abbreviation dictionary from all PubMed abstracts
import json
from collections import Counter, defaultdict

PUBMED_JSON = DATA_DIR / 'demo_pubmed.json'

with PUBMED_JSON.open('r', encoding='utf-8') as f:
    pubmed_records = json.load(f)

# Count candidate full names for each abbreviation across all abstracts.
abbr_full_counts = defaultdict(Counter)

for rec in pubmed_records[0:100]:
    abstract = normalize_spaces(str(rec.get('Abstract', '')))
    if not abstract:
        continue

    # Case 1: full phrase (ABBR)
    for m in ABBR_IN_PARENS.finditer(abstract):
        abbr = normalize_spaces(m.group('abbr'))
        if not is_abbreviation_like(abbr):
            continue

        full_name = refine_full_name_for_abbr(normalize_spaces(m.group('full')), abbr)
        if full_name:
            abbr_full_counts[abbr][full_name] += 1

    # Case 2: ABBR (full phrase)
    for m in FULL_IN_PARENS.finditer(abstract):
        abbr = normalize_spaces(m.group('abbr'))
        if not is_abbreviation_like(abbr):
            continue

        full_name = refine_full_name_for_abbr(normalize_spaces(m.group('full')), abbr)
        if full_name:
            abbr_full_counts[abbr][full_name] += 1

# Resolve to one value per key (most frequent full name)
abbr_to_full = {
    abbr: counts.most_common(1)[0][0]
    for abbr, counts in sorted(abbr_full_counts.items())
    if counts
}

# Optional: inspect abbreviations that mapped to multiple full names
abbr_conflicts = {
    abbr: dict(counts)
    for abbr, counts in sorted(abbr_full_counts.items())
    if len(counts) > 1
}

print('Total abbreviations:', len(abbr_to_full))
print('Abbreviations with multiple candidate full names:', len(abbr_conflicts))

# Final dictionary requested by user: key=abbreviation, value=full name
abbr_to_full

Total abbreviations: 14
Abbreviations with multiple candidate full names: 1


{'CNBD': 'channels are characterized by the presence of a carboxyl-terminal cyclic nucleotide-binding domain',
 'CNG': 'Cyclic nucleotide-gated',
 'CTL': 'Two synthetic O-GlcNAc-bearing peptides that elicit H-2Db-restricted glycopeptide-specific cytotoxic T cells',
 'GlcNAc': 'crystal structure of RSC-c in a complex with an N-acetylglucosamine tetramer,',
 'HCN': 'hyperpolarization-activated cyclic nucleotide-regulated',
 'K(d)': 'Isothermal titration calorimetry experiments measured the ligand binding affinity',
 'LBD2': 'we have determined the x-ray crystal structure of its ligand binding domain 2',
 'LUK': 'MesAT',
 'MesAT': 'strain LUK',
 'PLP': 'phosphate',
 'R': 'β-Phe,',
 'RSC-c': 'seeds',
 'S': 'we solved structures of MesAT in complex with',
 'TCR': 'T cell receptor'}

In [9]:
# (Debug) Commented-out duplicate save of the output CSV.
# result_df.to_csv(OUTPUT_CSV, index=False)
# print(f'Saved: {OUTPUT_CSV}')


In [10]:
# Extract per-abstract abbreviation/full-name pairs via regex and build a flat DataFrame.
# Build a dataframe with PubMed ID + abbreviation + full name (all matches)
abbr_rows = []

for rec in pubmed_records[0:100]:
    pmid = rec.get('PubMed_ID')
    abstract = normalize_spaces(str(rec.get('Abstract', '')))
    if not abstract:
        continue

    # Case 1: full phrase (ABBR)
    for m in ABBR_IN_PARENS.finditer(abstract):
        abbr = normalize_spaces(m.group('abbr'))
        if not is_abbreviation_like(abbr):
            continue

        full_name = refine_full_name_for_abbr(normalize_spaces(m.group('full')), abbr)
        if full_name:
            abbr_rows.append(
                {
                    'PubMed_ID': pmid,
                    'abbreviation': abbr,
                    'full_name': full_name,
                    'pattern_type': 'full_then_abbr',
                }
            )

    # Case 2: ABBR (full phrase)
    for m in FULL_IN_PARENS.finditer(abstract):
        abbr = normalize_spaces(m.group('abbr'))
        if not is_abbreviation_like(abbr):
            continue

        full_name = refine_full_name_for_abbr(normalize_spaces(m.group('full')), abbr)
        if full_name:
            abbr_rows.append(
                {
                    'PubMed_ID': pmid,
                    'abbreviation': abbr,
                    'full_name': full_name,
                    'pattern_type': 'abbr_then_full',
                }
            )

abbr_pubmed_df = pd.DataFrame(abbr_rows)
abbr_pubmed_df = abbr_pubmed_df.drop_duplicates().sort_values(
    ['abbreviation', 'PubMed_ID', 'full_name']
).reset_index(drop=True)

print('Total extracted abbreviation rows:', len(abbr_pubmed_df))
display(abbr_pubmed_df)

Total extracted abbreviation rows: 16


,PubMed_ID,abbreviation,full_name,pattern_type
0,28396445,CNBD,channels are characterized by the presence of a carboxyl-terminal cyclic nucleotide-binding domain,full_then_abbr
1,28396445,CNG,Cyclic nucleotide-gated,full_then_abbr
2,10023771,CTL,Two synthetic O-GlcNAc-bearing peptides that elicit H-2Db-restricted glycopeptide-specific cytotoxic T cells,full_then_abbr
3,22831795,GlcNAc,N HSQC spectrum of the stable isotope-labeled RSC-c were affected upon addition of,full_then_abbr
4,22831795,GlcNAc,Signal assignments revealed that most signals responsive to the addition of,full_then_abbr
5,22831795,GlcNAc,"crystal structure of RSC-c in a complex with an N-acetylglucosamine tetramer,",full_then_abbr
6,28396445,HCN,hyperpolarization-activated cyclic nucleotide-regulated,full_then_abbr
7,20053994,K(d),Isothermal titration calorimetry experiments measured the ligand binding affinity,full_then_abbr
8,20053994,LBD2,we have determined the x-ray crystal structure of its ligand binding domain 2,full_then_abbr
9,22745123,LUK,MesAT,abbr_then_full


In [11]:
# Configure and load the local Qwen model for LLM-based abbreviation extraction.
# Qwen extraction via local Hugging Face model (NO API CALLS)
# Outputs:
#   1) qwen_hf_abbr_to_full (dict)
#   2) qwen_hf_abbr_pubmed_df (DataFrame)

import json
import torch
from collections import Counter, defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

# Pick a free/open model from Hugging Face.
# You can switch to a smaller one if GPU/CPU memory is limited.
HF_QWEN_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

# Set to an int for quick test (e.g., 50), keep None for all abstracts.
HF_MAX_ABSTRACTS = None
HF_MAX_NEW_TOKENS = 256

SYSTEM_PROMPT_HF = (
    'You extract abbreviation-definition pairs from biomedical abstracts. '
    'Return strict JSON only with schema: {"pairs": [{"abbreviation": "...", "full_name": "..."}]}. '
    'Rules: '
    '1) abbreviation must not be pure numeric (e.g., 1, 2+, 420). '
    '2) keep nested abbreviation forms like M(pro), YPX(n)L. '
    '3) full_name should be concise noun phrase, not an entire sentence. '
    '4) if nothing found, return {"pairs": []}. '
)


def parse_qwen_json_pairs(content: str):
    """Parse model output into a list of {abbreviation, full_name}."""
    content = content.strip()

    # Handle fenced blocks
    if content.startswith('```'):
        content = content.strip('`')
        content = content.replace('json', '', 1).strip()

    try:
        payload = json.loads(content)
    except Exception:
        start = content.find('{')
        end = content.rfind('}')
        if start != -1 and end != -1 and end > start:
            payload = json.loads(content[start:end + 1])
        else:
            return []

    pairs = payload.get('pairs', []) if isinstance(payload, dict) else []
    out = []
    for p in pairs:
        if not isinstance(p, dict):
            continue
        abbr = normalize_spaces(str(p.get('abbreviation', '')))
        full = normalize_spaces(str(p.get('full_name', '')))
        if not abbr or not full:
            continue
        if not is_abbreviation_like(abbr):
            continue
        out.append({'abbreviation': abbr, 'full_name': full})
    return out


print(f'Loading Hugging Face model: {HF_QWEN_MODEL}')
tokenizer_hf = AutoTokenizer.from_pretrained(HF_QWEN_MODEL)
model_hf = AutoModelForCausalLM.from_pretrained(
    HF_QWEN_MODEL,
    torch_dtype='auto',
    device_map='auto',
)
model_hf.eval()


def extract_pairs_with_qwen_hf(abstract: str):
    """Local generation with Hugging Face Qwen model."""
    user_prompt = (
        'Extract abbreviation-definition pairs from this abstract.\\n\\n'
        f'Abstract:\\n{abstract}\\n\\n'
        'Return only JSON in this schema: '
        '{"pairs": [{"abbreviation": "...", "full_name": "..."}]}'
    )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_HF},
        {'role': 'user', 'content': user_prompt},
    ]

    prompt_text = tokenizer_hf.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer_hf([prompt_text], return_tensors='pt').to(model_hf.device)

    with torch.no_grad():
        outputs = model_hf.generate(
            **inputs,
            max_new_tokens=HF_MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    content = tokenizer_hf.decode(generated_ids, skip_special_tokens=True)
    return parse_qwen_json_pairs(content)


/home/kding1/anaconda3/envs/RAD_AI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Hugging Face model: Qwen/Qwen2.5-7B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.11it/s]


In [12]:
# Run Qwen over all PubMed abstracts to extract abbreviation–full-name pairs and collect results.
from tqdm import tqdm

records = pubmed_records if HF_MAX_ABSTRACTS is None else pubmed_records[:HF_MAX_ABSTRACTS]
records = records
qwen_hf_rows = []
qwen_hf_errors = []

for i, rec in tqdm(enumerate(records, start=1)):
    pmid = rec.get('PubMed_ID')
    abstract = normalize_spaces(str(rec.get('Abstract', '')))
    if not abstract:
        continue

    try:
        pairs = extract_pairs_with_qwen_hf(abstract)
        for pair in pairs:
            qwen_hf_rows.append(
                {
                    'PubMed_ID': pmid,
                    'abbreviation': pair['abbreviation'],
                    'full_name': pair['full_name'],
                }
            )
    except Exception as e:
        qwen_hf_errors.append({'PubMed_ID': pmid, 'error': str(e)})

    if i % 20 == 0:
        print(f'Processed {i}/{len(records)} abstracts ...')


# DataFrame output
qwen_hf_abbr_pubmed_df = pd.DataFrame(qwen_hf_rows)
if not qwen_hf_abbr_pubmed_df.empty:
    qwen_hf_abbr_pubmed_df = (
        qwen_hf_abbr_pubmed_df
        .drop_duplicates()
        .sort_values(['abbreviation', 'PubMed_ID', 'full_name'])
        .reset_index(drop=True)
    )

# Dictionary output (most frequent full name per abbreviation)
qwen_hf_counts = defaultdict(Counter)
for row in qwen_hf_rows:
    qwen_hf_counts[row['abbreviation']][row['full_name']] += 1

qwen_hf_abbr_to_full = {
    abbr: cnts.most_common(1)[0][0]
    for abbr, cnts in sorted(qwen_hf_counts.items())
    if cnts
}

print('HF-Qwen extracted rows:', len(qwen_hf_abbr_pubmed_df))
print('HF-Qwen unique abbreviations:', len(qwen_hf_abbr_to_full))
print('HF-Qwen errors:', len(qwen_hf_errors))

# Show outputs
display(qwen_hf_abbr_pubmed_df)
qwen_hf_abbr_to_full

OUTPUT_CSV = OUTPUT_DIR / 'abbr_dict_all_abs.csv'

qwen_hf_abbr_pubmed_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')


0it [00:00, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
5it [00:14,  2.86s/it]

HF-Qwen extracted rows: 16
HF-Qwen unique abbreviations: 16
HF-Qwen errors: 0


,PubMed_ID,abbreviation,full_name
0,22831795,(GlcNAc)(4),N-acetylglucosamine tetramer
1,28396445,CNBD,cyclic nucleotide-binding domain
2,28396445,CNG,cyclic nucleotide-gated
3,10023771,CTL,cytotoxic T cells
4,22831795,GH19,glycosyl hydrolase family 19
5,28396445,HCN,hyperpolarization-activated cyclic nucleotide-regulated
6,22831795,HPLC,high-performance liquid chromatography
7,22831795,HSQC,heteronuclear single quantum coherence
8,20053994,LBD2,ligand binding domain 2
9,10023771,MHC,major histocompatibility complex


Saved: /home/kding1/projects/RAD_PROTEIN/outputs/abbreviation/abbr_dict_all_abs.csv


In [13]:
# Save the Qwen-extracted abbreviation DataFrame to a CSV file.
qwen_hf_abbr_pubmed_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')


Saved: /home/kding1/projects/RAD_PROTEIN/outputs/abbreviation/abbr_dict_all_abs.csv


In [14]:
# Load and display the shipped abbreviation dictionary CSV.
import pandas as pd

df = pd.read_csv(ANNOTATION_DIR / 'abbr_dict_example.csv')
df

,PubMed_ID,abbreviation,full_name
0,17878296,CNG,cyclic nucleotide-gated
1,12968185,HCN,"hyperpolarization-activated, cyclic nucleotide-modulated"
2,20053994,LBD2,ligand binding domain 2
3,22831795,(GlcNAc)(4),N-acetylglucosamine tetramer
4,28396445,CNBD,cyclic nucleotide-binding domain
